# Lightweight Attention-Enhanced YOLOv8n for Pothole Detection (Res-CBAM)

**Upgraded, Kaggle-runnable pipeline.** Key fixes vs. the previous notebook:

1. **The attention module is now actually inside the trained network.** Previously, CBAM was defined and inserted into one model object, but the training cell reloaded a fresh `yolov8n.pt` — so the trained model contained **no attention at all**. Also, `insert()`-ing layers into `model.model.model` breaks YOLOv8's forward graph (layers are routed by index).
2. Attention is integrated the correct way: a **custom architecture YAML** (registered with Ultralytics' parser) placing **Res-CBAM after the P3/P4/P5 backbone stages**, with head skip-connections re-indexed accordingly.
3. **Pretrained weight transfer with index remapping** (naive `.load()` would only transfer the first few layers after insertion shifts indices).
4. **Identity-initialized residual attention** (`x + γ·CBAM(x)`, γ starts at 0) so fine-tuning starts from the intact pretrained representation — a defensible design point for the paper.
5. Reproducibility (fixed seed, deterministic), proper **baseline ablation option**, val **and test** evaluation, fixed metric-column names, attention heatmaps, fixed t-SNE, and an **efficiency benchmark** (params / GFLOPs / model size / FPS) to support the "lightweight" claim.

> ⚠️ Kaggle setup: enable **GPU (T4/P100)** and **Internet** in the notebook settings. Run cells in order.

In [ ]:
%pip install -q "ultralytics>=8.2.0"

import os, re, glob, time, random, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd
import yaml as pyyaml
import ultralytics
from ultralytics import YOLO

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("ultralytics:", ultralytics.__version__)
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

In [ ]:
CFG = dict(
    DATA_ROOT = "/kaggle/input/pothole-dataset-6knew",
    EPOCHS    = 150,
    IMGSZ     = 640,
    BATCH     = 16,
    PATIENCE  = 30,
    TRAIN_BASELINE = True,
)

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
DATA_YAML = os.path.join(WORK, "pothole_data.yaml")

def find_split(root, names):
    """Return the images dir for a split, handling both split/images and flat layouts."""
    for n in names:
        for cand in (os.path.join(root, n, "images"), os.path.join(root, n)):
            if os.path.isdir(cand) and glob.glob(os.path.join(cand, "*.[jp][pn]g")):
                return cand
    return None

train_dir = find_split(CFG["DATA_ROOT"], ["train"])
val_dir   = find_split(CFG["DATA_ROOT"], ["valid", "val"])
test_dir  = find_split(CFG["DATA_ROOT"], ["test"]) or val_dir

assert train_dir and val_dir, f"Could not find train/valid image folders under {CFG['DATA_ROOT']}. Check DATA_ROOT."

data_cfg = {"train": train_dir, "val": val_dir, "test": test_dir, "nc": 1, "names": ["pothole"]}
with open(DATA_YAML, "w") as f:
    pyyaml.safe_dump(data_cfg, f, sort_keys=False)

for k in ("train", "val", "test"):
    n = len(glob.glob(os.path.join(data_cfg[k], "*.[jp][pn]g")))
    print(f"{k:>5}: {n:5d} images  ->  {data_cfg[k]}")
print("Dataset YAML written to:", DATA_YAML)

In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, c1, reduction=16):
        super().__init__()
        hidden = max(c1 // reduction, 4)
        self.mlp = nn.Sequential(
            nn.Conv2d(c1, hidden, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, c1, 1, bias=False),
        )

    def forward(self, x):
        avg = self.mlp(F.adaptive_avg_pool2d(x, 1))
        mx  = self.mlp(F.adaptive_max_pool2d(x, 1))
        return x * torch.sigmoid(avg + mx)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.last_map = None
        self.store_maps = False

    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        sa = torch.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))
        if self.store_maps:
            self.last_map = sa.detach().float().cpu()
        return x * sa

class ResCBAM(nn.Module):
    """Residual CBAM: out = x + gamma * SA(CA(x)), gamma initialized to 0 (identity at init)."""
    def __init__(self, c1, kernel_size=7, reduction=16):
        super().__init__()
        self.ca = ChannelAttention(c1, reduction)
        self.sa = SpatialAttention(kernel_size)
        self.gamma = nn.Parameter(torch.zeros(1))

    @property
    def store_maps(self):
        return self.sa.store_maps

    @store_maps.setter
    def store_maps(self, v):
        self.sa.store_maps = v

    @property
    def last_map(self):
        return self.sa.last_map

    def forward(self, x):
        return x + self.gamma * self.sa(self.ca(x))

import ultralytics.nn.tasks as _tasks
_tasks.ResCBAM = ResCBAM
print("ResCBAM registered with the Ultralytics parser.")

In [ ]:
ARCH_YAML = os.path.join(WORK, "yolov8n_rescbam.yaml")

arch_text = """
nc: 1
depth_multiple: 1.0
width_multiple: 1.0

backbone:
  - [-1, 1, Conv,    [16, 3, 2]]
  - [-1, 1, Conv,    [32, 3, 2]]
  - [-1, 1, C2f,     [32, True]]
  - [-1, 1, Conv,    [64, 3, 2]]
  - [-1, 2, C2f,     [64, True]]
  - [-1, 1, ResCBAM, [64, 7]]
  - [-1, 1, Conv,    [128, 3, 2]]
  - [-1, 2, C2f,     [128, True]]
  - [-1, 1, ResCBAM, [128, 7]]
  - [-1, 1, Conv,    [256, 3, 2]]
  - [-1, 1, C2f,     [256, True]]
  - [-1, 1, ResCBAM, [256, 7]]
  - [-1, 1, SPPF,    [256, 5]]

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 8], 1, Concat, [1]]
  - [-1, 1, C2f, [128]]
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 5], 1, Concat, [1]]
  - [-1, 1, C2f, [64]]
  - [-1, 1, Conv, [64, 3, 2]]
  - [[-1, 15], 1, Concat, [1]]
  - [-1, 1, C2f, [128]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 1, C2f, [256]]
  - [[18, 21, 24], 1, Detect, [nc]]"""
with open(ARCH_YAML, "w") as f:
    f.write(arch_text)

model_prop = YOLO(ARCH_YAML, task="detect")
model_prop.load("yolov8n.pt")

OLD2NEW = {0:0, 1:1, 2:2, 3:3, 4:4, 5:6, 6:7, 7:9, 8:10, 9:12, 10:13, 11:14, 12:15,
           13:16, 14:17, 15:18, 16:19, 17:20, 18:21, 19:22, 20:23, 21:24, 22:25}

src_sd = YOLO("yolov8n.pt").model.state_dict()
dst_sd = model_prop.model.state_dict()
remapped = {}
for k, v in src_sd.items():
    m = re.match(r"^model\.(\d+)\.(.*)$", k)
    if m and int(m.group(1)) in OLD2NEW:
        nk = f"model.{OLD2NEW[int(m.group(1))]}.{m.group(2)}"
        if nk in dst_sd and dst_sd[nk].shape == v.shape:
            remapped[nk] = v
missing = model_prop.model.load_state_dict(remapped, strict=False)
print(f"Transferred {len(remapped)}/{len(dst_sd)} tensors from COCO-pretrained yolov8n "
      f"(the rest are Res-CBAM blocks + nc-dependent Detect convs, trained from scratch).")

n_cbam = sum(isinstance(m, ResCBAM) for m in model_prop.model.modules())
assert n_cbam == 3, f"Expected 3 ResCBAM blocks, found {n_cbam}"
p_prop = sum(p.numel() for p in model_prop.model.parameters())
p_base = sum(p.numel() for p in YOLO("yolov8n.pt").model.parameters())
print(f"ResCBAM blocks in model: {n_cbam}")
print(f"Params  | baseline yolov8n: {p_base/1e6:.3f}M   proposed: {p_prop/1e6:.3f}M   "
      f"(+{(p_prop-p_base)/1e3:.1f}K, +{100*(p_prop-p_base)/p_base:.2f}%)")
model_prop.info()

## Training

Identical hyperparameters are used for the proposed model and the baseline so the ablation is fair (this matters for reviewers). `seed=42, deterministic=True` for reproducibility. Keep `device=0` (single GPU) — multi-GPU DDP spawns subprocesses that don't have the custom `ResCBAM` module registered.

In [ ]:
TRAIN_ARGS = dict(
    data=DATA_YAML, epochs=CFG["EPOCHS"], imgsz=CFG["IMGSZ"], batch=CFG["BATCH"],
    device=DEVICE, seed=SEED, deterministic=True, patience=CFG["PATIENCE"],
    optimizer="auto", cos_lr=True, plots=True,
    project=os.path.join(WORK, "runs"), exist_ok=True,
)

results_prop = model_prop.train(name="yolov8n_rescbam", **TRAIN_ARGS)

PROP_DIR  = str(model_prop.trainer.save_dir)
BEST_PROP = os.path.join(PROP_DIR, "weights", "best.pt")
print("Run dir :", PROP_DIR)
print("Best wts:", BEST_PROP)

_chk = YOLO(BEST_PROP)
print("ResCBAM blocks in best.pt:", sum(isinstance(m, ResCBAM) for m in _chk.model.modules()))
gammas = [float(m.gamma.detach().cpu()) for m in _chk.model.modules() if isinstance(m, ResCBAM)]
print("Learned attention gates gamma (P3, P4, P5):", [round(g, 4) for g in gammas])
del _chk

In [ ]:
BEST_BASE, BASE_DIR = None, None
if CFG["TRAIN_BASELINE"]:
    model_base = YOLO("yolov8n.pt")
    results_base = model_base.train(name="yolov8n_baseline", **TRAIN_ARGS)
    BASE_DIR  = str(model_base.trainer.save_dir)
    BEST_BASE = os.path.join(BASE_DIR, "weights", "best.pt")
    print("Baseline run dir:", BASE_DIR)
else:
    print("Baseline training skipped (CFG['TRAIN_BASELINE'] = False).")

In [ ]:
def evaluate(weights, label):
    m = YOLO(weights)
    rows = []
    for split in ("val", "test"):
        r = m.val(data=DATA_YAML, split=split, imgsz=CFG["IMGSZ"], device=DEVICE,
                  plots=(split == "val"), verbose=False)
        d = r.results_dict
        rows.append({
            "model": label, "split": split,
            "precision": d.get("metrics/precision(B)"),
            "recall":    d.get("metrics/recall(B)"),
            "mAP50":     d.get("metrics/mAP50(B)"),
            "mAP50-95":  d.get("metrics/mAP50-95(B)"),
        })
    return rows

rows = evaluate(BEST_PROP, "YOLOv8n + Res-CBAM (proposed)")
if BEST_BASE:
    rows += evaluate(BEST_BASE, "YOLOv8n (baseline)")

metrics_df = pd.DataFrame(rows).round(4)
metrics_df.to_csv(os.path.join(WORK, "metrics_summary.csv"), index=False)
print("Saved:", os.path.join(WORK, "metrics_summary.csv"))
metrics_df

In [ ]:
def plot_curves(run_dir, label):
    csv_path = os.path.join(run_dir, "results.csv")
    if not os.path.exists(csv_path):
        print("No results.csv in", run_dir); return
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    ep = df["epoch"] if "epoch" in df.columns else np.arange(len(df))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for c in ("train/box_loss", "val/box_loss", "train/cls_loss", "val/cls_loss",
              "train/dfl_loss", "val/dfl_loss"):
        if c in df.columns:
            axes[0].plot(ep, df[c], label=c)
    axes[0].set(xlabel="Epoch", ylabel="Loss", title=f"Losses — {label}"); axes[0].legend(fontsize=8)

    for c in ("metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"):
        if c in df.columns:
            axes[1].plot(ep, df[c], label=c)
    axes[1].set(xlabel="Epoch", ylabel="Value", title=f"Metrics — {label}"); axes[1].legend(fontsize=8)

    out = os.path.join(WORK, f"curves_{label.replace(' ', '_')}.png")
    plt.tight_layout(); plt.savefig(out, dpi=200); plt.show()
    print("Saved:", out)

plot_curves(PROP_DIR, "proposed")
if BASE_DIR:
    plot_curves(BASE_DIR, "baseline")

for f in ("PR_curve.png", "F1_curve.png", "confusion_matrix.png", "confusion_matrix_normalized.png"):
    p = os.path.join(PROP_DIR, f)
    if os.path.exists(p): print("Journal figure available:", p)

In [ ]:
model_best = YOLO(BEST_PROP)
test_imgs = sorted(glob.glob(os.path.join(data_cfg["test"], "*.[jp][pn]g")))
sample = test_imgs[:: max(1, len(test_imgs) // 6)][:6]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, p in zip(axes.ravel(), sample):
    r = model_best.predict(p, imgsz=CFG["IMGSZ"], conf=0.25, device=DEVICE, verbose=False)[0]
    ax.imshow(r.plot()[:, :, ::-1])
    ax.set_title(os.path.basename(p), fontsize=8); ax.axis("off")
plt.suptitle("Proposed model — test-set predictions")
plt.tight_layout()
plt.savefig(os.path.join(WORK, "qualitative_predictions.png"), dpi=200)
plt.show()

In [ ]:
from PIL import Image

cbams = [m for m in model_best.model.modules() if isinstance(m, ResCBAM)]
for c in cbams:
    c.store_maps = True

viz_imgs = sample[:3]
fig, axes = plt.subplots(len(viz_imgs), 4, figsize=(18, 4.5 * len(viz_imgs)))
axes = np.atleast_2d(axes)
level_names = ["P3 (stride 8)", "P4 (stride 16)", "P5 (stride 32)"]

for row, p in enumerate(viz_imgs):
    _ = model_best.predict(p, imgsz=CFG["IMGSZ"], device=DEVICE, verbose=False)
    img = np.array(Image.open(p).convert("RGB"))
    axes[row, 0].imshow(img); axes[row, 0].set_title("Input"); axes[row, 0].axis("off")
    for col, (c, name) in enumerate(zip(cbams, level_names), start=1):
        sa = c.last_map
        sa_up = F.interpolate(sa, size=img.shape[:2], mode="bilinear", align_corners=False)
        axes[row, col].imshow(img)
        axes[row, col].imshow(sa_up[0, 0].numpy(), cmap="jet", alpha=0.45)
        axes[row, col].set_title(f"Res-CBAM {name}"); axes[row, col].axis("off")

for c in cbams:
    c.store_maps = False

plt.tight_layout()
plt.savefig(os.path.join(WORK, "attention_heatmaps.png"), dpi=200)
plt.show()

In [ ]:
from sklearn.manifold import TSNE

sppf = [m for m in model_best.model.modules() if m.__class__.__name__ == "SPPF"][-1]
feats = []
hook = sppf.register_forward_hook(
    lambda mod, inp, out: feats.append(F.adaptive_avg_pool2d(out.detach().float(), 1)
                                       .flatten(1).cpu().numpy()[0]))

val_imgs = sorted(glob.glob(os.path.join(data_cfg["val"], "*.[jp][pn]g")))[:300]
label_dir = data_cfg["val"].replace(os.sep + "images", os.sep + "labels")
labels = []
for p in val_imgs:
    _ = model_best.predict(p, imgsz=CFG["IMGSZ"], device=DEVICE, verbose=False)
    lp = os.path.join(label_dir, os.path.splitext(os.path.basename(p))[0] + ".txt")
    has_obj = os.path.exists(lp) and os.path.getsize(lp) > 0
    labels.append(1 if has_obj else 0)
hook.remove()

X = np.stack(feats); y = np.array(labels)
emb = TSNE(n_components=2, random_state=SEED,
           perplexity=min(30, len(X) - 1)).fit_transform(X)

plt.figure(figsize=(8, 6))
for cls, name in [(0, "Background"), (1, "Pothole")]:
    idx = y == cls
    plt.scatter(emb[idx, 0], emb[idx, 1], s=14, alpha=0.6, label=f"{name} (n={idx.sum()})")
plt.legend(); plt.title("t-SNE of SPPF features — validation set")
plt.xlabel("t-SNE 1"); plt.ylabel("t-SNE 2")
plt.savefig(os.path.join(WORK, "tsne_features.png"), dpi=200)
plt.show()

In [ ]:
def benchmark(weights, label, n_warmup=10, n_runs=50):
    m = YOLO(weights)
    params = sum(p.numel() for p in m.model.parameters())
    try:
        flops = m.info(verbose=False)[-1]
    except Exception:
        flops = float("nan")
    size_mb = os.path.getsize(weights) / 1e6
    img = val_imgs[0]
    for _ in range(n_warmup):
        m.predict(img, imgsz=CFG["IMGSZ"], device=DEVICE, verbose=False)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n_runs):
        m.predict(img, imgsz=CFG["IMGSZ"], device=DEVICE, verbose=False)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) / n_runs * 1000
    return {"model": label, "params (M)": round(params / 1e6, 3), "GFLOPs": flops,
            "size (MB)": round(size_mb, 2), "latency (ms/img)": round(ms, 2),
            "FPS": round(1000 / ms, 1)}

bench_rows = [benchmark(BEST_PROP, "YOLOv8n + Res-CBAM (proposed)")]
if BEST_BASE:
    bench_rows.append(benchmark(BEST_BASE, "YOLOv8n (baseline)"))

bench_df = pd.DataFrame(bench_rows)
bench_df.to_csv(os.path.join(WORK, "efficiency_benchmark.csv"), index=False)
print("Note: report the hardware (e.g., Kaggle T4) alongside latency/FPS in the paper.")
bench_df

## Notes for the paper (read before submitting)

**What you can now claim honestly.** The trained network genuinely contains three identity-initialized residual CBAM blocks at P3/P4/P5 (verified programmatically after training), with COCO-pretrained initialization via index-remapped weight transfer. The learned γ values printed after training show *how much* attention the network chose to use at each scale — that itself is a nice analysis point.

**On novelty — be careful.** "YOLOv8 + CBAM for pothole detection" alone has been published many times and is unlikely to clear a Q1 bar by itself. What strengthens this work:

- The **residual identity-initialized gating** (γ-gated attention preserving pretrained features) and the **γ analysis** across scales — frame this as your modification, and ablate it (Res-CBAM vs plain CBAM vs baseline).
- **Placement ablation**: attention at P3 only vs P3/P4/P5 vs all stages (edit the YAML; small-object detection often benefits most at P3).
- **Efficiency table** (params/GFLOPs/FPS) proving the "lightweight" claim quantitatively.
- Evaluate on a **second public dataset** (e.g., RDD2022) for generalization — Q1 reviewers almost always ask for this.
- Report **mean ± std over 3 seeds** if GPU budget allows; single-run deltas of 1–2 mAP points are within noise.

**Reproducing/reloading:** `best.pt` pickles the `ResCBAM` class by reference — in any new session, run the setup and module cells (1–4) before `YOLO(best.pt)`.

**Outputs written to `/kaggle/working/`:** `metrics_summary.csv`, `efficiency_benchmark.csv`, `curves_*.png`, `qualitative_predictions.png`, `attention_heatmaps.png`, `tsne_features.png`, plus full Ultralytics run dirs under `runs/` (PR/F1 curves, confusion matrices, `best.pt`).